### Purpose
This notebook builds the hybrid supervised model by combining the engineered sales features with anomaly signals generated in the anomaly detection step.
<br> --- --- --- <br>
Este notebook constrói o modelo supervisionado híbrido combinando as features de vendas com os sinais de anomalia gerados na etapa de detecção de anomalias.
<br>
<br>
📝 Language: Technical documentation is maintained in English to ensure consistency and ease of maintenance. Bilingual support (Portuguese/English) is available exclusively within the notebooks.
<br> --- --- --- <br>
📝 Idioma: A documentação é mantida em inglês para garantir consistência técnica e evitar retrabalho na documentação. Isto aplica-se somente aos notebooks.

### Main idea
The notebook merges the feature dataset with the anomaly prediction dataset and trains a LightGBM regression model that uses anomaly-related information as additional input.
<br> --- --- --- <br>
O notebook faz o merge entre o dataset de features e o dataset de predições de anomalia e treina um modelo de regressão LightGBM que usa informações de anomalia como entrada adicional.

##### ⚙️The project follows this logic:
**raw data -> data quality -> feature engineering -> anomaly detection -> baseline model -> hybrid model -> performance benchmarking**

##### ⚙️Execution order
📝01_data_loading.ipynb -> 📝02_data_quality.ipynb -> 📝03_feature_engineering.ipynb -> 📝04_anomaly_detection.ipynb -> 
📝05_BaseLine.ipynb -> 📝06_HybridModel.ipynb -> 📝07_Interpretability_SHAP_Analysis.ipynb -> 📝08_performance_benchmarking.ipynb 


### Inputs
- 📦`beverage_sales_feature.parquet`
- 📦`anomaly_predictions.parquet`

### Main processing steps
1. Load engineered features and anomaly predictions.
2. Convert `Order_Date` to datetime.
3. Merge both datasets using:
   - `Order_Date`
   - `Product`
   - `Region`
4. Split the data by year:
   - 2021–2022 for training
   - 2023 for test or later evaluation
5. Define:
   - target column: `quantity_sum`
   - numeric feature list
   - categorical feature list
6. Train the hybrid LightGBM regressor.
7. Evaluate model performance.
8. Compare prediction error by `anomaly_flag`.
9. Save the trained model and the best parameters.
<br>
<br> --- --- --- <br>

1. Carregar as features engenheiradas e as predições de anomalia.
2. Converter `Order_Date` para datetime.
3. Fazer o merge dos dois datasets usando:
   - `Order_Date`
   - `Product`
   - `Region`
4. Separar os dados por ano:
   - 2021–2022 para treino
   - 2023 para teste ou avaliação posterior
5. Definir:
   - coluna alvo: `quantity_sum`
   - lista de features numéricas
   - lista de features categóricas
6. Treinar o regressor híbrido com LightGBM.
7. Avaliar o desempenho do modelo.
8. Comparar o erro de predição por `anomaly_flag`.
9. Salvar o modelo treinado e os melhores parâmetros.

### Outputs
- Trained hybrid model file
- JSON file with best parameters / metrics
- Evaluation metrics
- Error comparison grouped by anomaly flag
<br>
<br> --- --- --- <br>

- Arquivo do modelo híbrido treinado
- Arquivo JSON com melhores parâmetros / métricas
- Métricas de avaliação
- Comparação de erro agrupada por flag de anomalia

### Why this notebook matters
This notebook is important because it connects the unsupervised stage with the supervised stage.  
Instead of using anomaly detection only for monitoring, the project uses anomaly signals as extra information for the regression model.
<br> --- --- --- <br>
Este notebook é importante porque conecta a etapa não supervisionada com a etapa supervisionada.  
Em vez de usar a detecção de anomalias apenas para monitoramento, o projeto usa os sinais de anomalia como informação extra para o modelo de regressão.

### 💡Insights
The hybrid approach allows the model to learn not only from historical sales patterns, but also from abnormal behavior detected earlier in the pipeline. 
<br> --- --- --- <br>
A abordagem híbrida permite que o modelo aprenda não apenas com padrões históricos de vendas, mas também com comportamentos anormais detectados antes no pipeline. 

### Notes
- Since the target is `quantity_sum`, this notebook is focused on aggregated demand prediction rather than raw transaction-level quantity.
<br> --- --- --- <br>

- Como o alvo é `quantity_sum`, este notebook está focado em previsão de demanda agregada e não na quantidade bruta por transação.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import glob
from pathlib import Path
import sys
import pyarrow
import pandas as pd


PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

from src.config.config import DATA_PROCESSED, DATA_FEATURES, MODELS_BASELINE,MODELS_METRICS
from src.repository.parquet_repository import ParquetRepository
from src.models.LightGBMRegressorBaseLine import LightGBMRegressorBaseLine
from src.models.LightGBMRegressorAnomaly import LightGBMRegressorAnomaly



In [2]:
repo = ParquetRepository(DATA_FEATURES)

In [3]:
df_features = repo.load("beverage_sales_feature.parquet")
df_anomalies = repo.load("anomaly_predictions.parquet")

df_features["Order_Date"] = pd.to_datetime(df_features["Order_Date"])
df_anomalies["Order_Date"] = pd.to_datetime(df_anomalies["Order_Date"])

[OK] Arquivo carregado: D:\PROJETOS\git_repo\BEVERAGE-SALES\data\features\beverage_sales_feature.parquet
[OK] Arquivo carregado: D:\PROJETOS\git_repo\BEVERAGE-SALES\data\features\anomaly_predictions.parquet


In [4]:
print (df_anomalies.dtypes)

Order_Date                  datetime64[ns]
Category                            object
Product                             object
Region                              object
quantity_sum                       float64
total_price_sum                    float64
unit_price_mean                    float64
discount_mean                      float64
order_count                        float64
customer_count                     float64
avg_ticket                         float64
day_of_week                          int32
month                                int32
year                                 int32
is_weekend                           int64
quantity_sum_mean_7d               float64
quantity_sum_std_7d                float64
quantity_sum_sum_7d                float64
total_price_sum_mean_7d            float64
total_price_sum_std_7d             float64
total_price_sum_mean_30d           float64
unit_price_mean_mean_7d            float64
discount_mean_mean_14d             float64
quantity_vs

In [5]:
df_model_with_anomaly = df_features.merge(
    df_anomalies,
    on=["Order_Date", "Product", "Region"],
    how="left"
)

In [6]:
print (df_anomalies.dtypes)

Order_Date                  datetime64[ns]
Category                            object
Product                             object
Region                              object
quantity_sum                       float64
total_price_sum                    float64
unit_price_mean                    float64
discount_mean                      float64
order_count                        float64
customer_count                     float64
avg_ticket                         float64
day_of_week                          int32
month                                int32
year                                 int32
is_weekend                           int64
quantity_sum_mean_7d               float64
quantity_sum_std_7d                float64
quantity_sum_sum_7d                float64
total_price_sum_mean_7d            float64
total_price_sum_std_7d             float64
total_price_sum_mean_30d           float64
unit_price_mean_mean_7d            float64
discount_mean_mean_14d             float64
quantity_vs

In [7]:
df_anomalies["Order_Date"] = pd.to_datetime(df_anomalies["Order_Date"])

df_train_anomaly = df_anomalies[df_anomalies["Order_Date"].dt.year.isin([2021, 2022])].copy()
df_test_anomaly = df_anomalies[df_anomalies["Order_Date"].dt.year == 2023].copy()

In [8]:
print (df_test_anomaly.dtypes)

Order_Date                  datetime64[ns]
Category                            object
Product                             object
Region                              object
quantity_sum                       float64
total_price_sum                    float64
unit_price_mean                    float64
discount_mean                      float64
order_count                        float64
customer_count                     float64
avg_ticket                         float64
day_of_week                          int32
month                                int32
year                                 int32
is_weekend                           int64
quantity_sum_mean_7d               float64
quantity_sum_std_7d                float64
quantity_sum_sum_7d                float64
total_price_sum_mean_7d            float64
total_price_sum_std_7d             float64
total_price_sum_mean_30d           float64
unit_price_mean_mean_7d            float64
discount_mean_mean_14d             float64
quantity_vs

In [9]:
target_col = "quantity_sum"

numeric_features = [
    "total_price_sum",
    "unit_price_mean",
    "discount_mean",
    "order_count",
    "customer_count",
    "avg_ticket",
    "day_of_week",
    "month",
    "is_weekend",
    "quantity_sum_mean_7d",
    "quantity_sum_std_7d",
    "quantity_sum_sum_7d",
    "total_price_sum_mean_7d",
    "total_price_sum_std_7d",
    "total_price_sum_mean_30d",
    "unit_price_mean_mean_7d",
    "discount_mean_mean_14d",
    "quantity_vs_mean_7d",
    "total_price_vs_mean_30d",
    "quantity_pct_vs_mean_7d",
    "history_less_than_7d",
    "history_less_than_30d",
    "anomaly_score",
    "anomaly_flag"
]

categorical_features = [
    "Product",
    "Region"
]

anomaly_model = LightGBMRegressorAnomaly(
    target_col=target_col,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    model_dir=MODELS_BASELINE,
    random_state=3,
    n_splits=3,
    scoring="neg_root_mean_squared_error"
)

grid_result = anomaly_model.fit(df_train_anomaly)

Fitting 3 folds for each of 48 candidates, totalling 144 fits


BrokenProcessPool: A task has failed to un-serialize. Please ensure that the arguments of the function are all picklable.

In [ ]:
metrics = anomaly_model.evaluate(df_anomalies)

print(metrics)

In [ ]:
df_eval = df_anomalies.copy()

df_eval["y_true"] = df_eval["quantity_sum"]
df_eval["y_pred"] = anomaly_model.predict(df_eval)
df_eval["abs_error"] = abs(df_eval["y_true"] - df_eval["y_pred"])

df_eval.groupby("anomaly_flag")["abs_error"].agg(
    count="count",
    mean_abs_error="mean",
    median_abs_error="median",
    max_abs_error="max"
)

In [ ]:
model_path = anomaly_model.save_model(filename="lightgbm_anomalies_model.joblib")
params_path = anomaly_model.save_best_params(filename="lightgbm_anomalies_metrics.json")

print("Model saved at:", model_path)
print("Best params saved at:", params_path)
